In [2]:
import sys
sys.path.append("..")

from auto_pricer.car import Car

HF_USER = "ShahidHKhan"  # confirm this matches your actual HF username from the instructions doc

train, val, test = Car.from_hub(f"{HF_USER}/cars_lite_processed")

print(f"Test set size: {len(test)}")
print(test[0])
print()
print("Summary:", test[0].summary)
print("Actual price:", test[0].price)

Test set size: 1452
title='2020 Kia Soul' make='Kia' model='Soul' trim='LX FWD' year=2020 mileage=4823.0 price=15995.0 horsepower=147.0 body_type='SUV / Crossover' fuel_type='Gasoline' transmission='Continuously Variable Transmission' has_accidents=False frame_damaged=False full=None summary='Title: 2020 Kia Soul LX\nCategory: SUV\nMake: Kia\nDescription: Excellent condition, one-owner, nonsmoker vehicle with no reported accidents.\nDetails: 4,823 miles, gasoline-powered with a continuously variable transmission.' prompt=None completion=None id=None

Summary: Title: 2020 Kia Soul LX
Category: SUV
Make: Kia
Description: Excellent condition, one-owner, nonsmoker vehicle with no reported accidents.
Details: 4,823 miles, gasoline-powered with a continuously variable transmission.
Actual price: 15995.0


In [3]:
from litellm import completion

def messages_for(car):
    return [{"role": "user", "content": f"Estimate the price of this used car. Respond with the price, no explanation\n\n{car.summary}"}]

def gemini_2_5_flash(car):
    response = completion(model="gemini/gemini-2.5-flash", messages=messages_for(car))
    return response.choices[0].message.content

# single-row test
test_car = test[0]
result = gemini_2_5_flash(test_car)
print("Prediction (raw):", repr(result))
print("Actual price:", test_car.price)

Prediction (raw): '$17,800'
Actual price: 15995.0


In [4]:
import re

def parse_price(text):
    """Extract a numeric price from Gemini's response, e.g. '$17,800' -> 17800.0"""
    if text is None:
        return 0.0
    match = re.search(r'[\d,]+\.?\d*', text.replace('$', '').replace(',', ''))
    return float(match.group()) if match else 0.0

# test it on what we already got
print(parse_price(result))  # should print 17800.0

17800.0


In [5]:
from litellm import completion

def gemini_2_5_flash_tracked(car):
    response = completion(model="gemini/gemini-2.5-flash", messages=messages_for(car))
    cost = response._hidden_params.get("response_cost", 0)
    return response.choices[0].message.content, cost

sample = test[:10]
total_cost = 0
results = []

for car in sample:
    raw, cost = gemini_2_5_flash_tracked(car)
    pred = parse_price(raw)
    total_cost += cost
    error = abs(pred - car.price)
    results.append((car.title, car.price, pred, error))
    print(f"{car.title[:40]:40s} actual=${car.price:>9,.0f}  pred=${pred:>9,.0f}  error=${error:>8,.0f}")

print(f"\nTotal cost for 10 cars: ${total_cost:.5f}")
print(f"Avg cost per car: ${total_cost/10:.5f}")
print(f"Projected cost for full test set ({len(test)} cars): ${(total_cost/10)*len(test):.4f}")

2020 Kia Soul                            actual=$   15,995  pred=$   17,300  error=$   1,305
2014 Lexus LX 570                        actual=$   34,894  pred=$   36,500  error=$   1,606
2017 Jeep Wrangler Unlimited             actual=$   30,497  pred=$   29,000  error=$   1,497
2011 Toyota Corolla                      actual=$    7,595  pred=$    7,700  error=$     105
2017 Ford Edge                           actual=$   20,896  pred=$   23,900  error=$   3,004
2011 Nissan Murano                       actual=$    9,545  pred=$    8,900  error=$     645
2018 Acura RDX                           actual=$   25,905  pred=$   23,500  error=$   2,405
2007 Subaru B9 Tribeca                   actual=$    9,575  pred=$    2,750  error=$   6,825
2017 Ford Explorer                       actual=$   27,495  pred=$   25,750  error=$   1,745
2015 Jeep Grand Cherokee                 actual=$   18,995  pred=$   21,500  error=$   2,505

Total cost for 10 cars: $0.01970
Avg cost per car: $0.00197
Projected

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def run_zero_shot_batch(cars, model_fn, max_workers=8):
    results = [None] * len(cars)
    total_cost = [0]
    pbar = tqdm(total=len(cars))

    def run_one(i, car):
        raw, cost = model_fn(car)
        pred = parse_price(raw)
        total_cost[0] += cost
        pbar.update(1)
        return i, pred

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(run_one, i, car) for i, car in enumerate(cars)]
        for f in as_completed(futures):
            i, pred = f.result()
            results[i] = pred

    pbar.close()
    print(f"Total cost: ${total_cost[0]:.4f}")
    return results

flash_predictions = run_zero_shot_batch(test, gemini_2_5_flash_tracked, max_workers=8)

100%|██████████| 1452/1452 [15:57<00:00,  1.52it/s]

Total cost: $3.5227


In [9]:
import numpy as np

actuals = np.array([car.price for car in test])
preds = np.array(flash_predictions)
errors = np.abs(preds - actuals)

print(f"Mean Absolute Error: ${errors.mean():,.2f}")
print(f"Median Absolute Error: ${np.median(errors):,.2f}")
print(f"Max error: ${errors.max():,.2f}")
print(f"Min error: ${errors.min():,.2f}")

# how many predictions failed to parse (returned 0.0)?
zero_preds = (preds == 0.0).sum()
print(f"\nPredictions that failed to parse (0.0): {zero_preds}")

Mean Absolute Error: $2,996.45
Median Absolute Error: $2,082.50
Max error: $33,100.00
Min error: $0.00

Predictions that failed to parse (0.0): 0


In [10]:
worst_idx = np.argmax(errors)
worst_car = test[worst_idx]

print("Summary:", worst_car.summary)
print(f"Actual price: ${worst_car.price:,.2f}")
print(f"Predicted price: ${preds[worst_idx]:,.2f}")
print(f"Error: ${errors[worst_idx]:,.2f}")

Summary: Title: 2014 Aston Martin Vanquish Coupe
Category: Coupe
Make: Aston Martin
Description: Exceptionally well-maintained 2014 Aston Martin Vanquish Coupe with low mileage and no reported accidents.
Details: Features 14,397 miles, gasoline fuel, and a 6-speed automatic transmission.
Actual price: $116,900.00
Predicted price: $150,000.00
Error: $33,100.00


In [11]:
def gemini_2_5_pro_tracked(car):
    response = completion(model="gemini/gemini-2.5-pro", messages=messages_for(car))
    cost = response._hidden_params.get("response_cost", 0)
    return response.choices[0].message.content, cost

pro_sample = test[:50]
pro_predictions = []
pro_total_cost = 0

for car in pro_sample:
    raw, cost = gemini_2_5_pro_tracked(car)
    pred = parse_price(raw)
    pro_total_cost += cost
    pro_predictions.append(pred)

pro_actuals = np.array([car.price for car in pro_sample])
pro_preds = np.array(pro_predictions)
pro_errors = np.abs(pro_preds - pro_actuals)

print(f"Total cost for 50 cars: ${pro_total_cost:.4f}")
print(f"Mean Absolute Error: ${pro_errors.mean():,.2f}")
print(f"Median Absolute Error: ${np.median(pro_errors):,.2f}")
print(f"Max error: ${pro_errors.max():,.2f}")
zero_preds = (pro_preds == 0.0).sum()
print(f"Parse failures: {zero_preds}")

Total cost for 50 cars: $0.6848
Mean Absolute Error: $3,326.40
Median Absolute Error: $2,492.50
Max error: $15,498.00
Parse failures: 0
